<a href="https://colab.research.google.com/github/emazarchie-cpu/Emmanuel-Archibong-ML-Intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emazarchie-cpu/Emmanuel-Archibong-ML-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

dataset = HfApi().dataset_info(
    repo_id="FlyRank/internship-warehouse",
    token=HF_TOKEN
)

print("Repository:", dataset.id)
print("Dataset accessible:", dataset.id == "FlyRank/internship-warehouse")

Repository: FlyRank/internship-warehouse
Dataset accessible: True


In [ ]:
from huggingface_hub import HfApi

try:
    info = HfApi().dataset_info(
        repo_id="FlyRank/internship-warehouse",
        token=HF_TOKEN
    )
    print("Dataset access successful!")
    print("Dataset:", info.id)
except Exception as e:
    print("Dataset access check failed:", type(e).__name__)
    print(str(e))

Dataset access successful!
Dataset: FlyRank/internship-warehouse


In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

try:
    user = HfApi().whoami(token=HF_TOKEN)
    print("Authentication successful!")
    print("Hugging Face username:", user["name"])
except Exception as e:
    print("Authentication failed:", type(e).__name__)
    print(str(e))

Authentication successful!
Hugging Face username: emazarchie


In [ ]:
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
    print("HF_TOKEN retrieved successfully:", bool(token))
except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error message:", str(e))

HF_TOKEN retrieved successfully: True


In [ ]:
%pip install -q duckdb huggingface_hub

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Hugging Face token loaded successfully!")

Hugging Face token loaded successfully!


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing. Check your Colab Secrets settings.")

import os
os.environ["HF_TOKEN"] = HF_TOKEN

print("Hugging Face authentication token loaded.")

Hugging Face authentication token loaded.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis and Time Window

**Unit of analysis:** One content item belonging to one client. Daily performance records will be aggregated to the content item level before analysis.

**Feature window:** 1–28 February 2026. Features will be calculated from Google Search Console (GSC) performance observed during February.

**Outcome window:** 1–31 March 2026. March performance will be used to measure subsequent outcomes, including click-through rate (CTR) and average search position.

**Decision cutoff:** 28 February 2026. No information observed after this cutoff may be used as a predictive feature.

**Minimum data requirement:** Content items must have at least 100 GSC impressions during March to qualify for the outcome analysis.

**Intended use:** The resulting dataset will support editorial-review prioritization. The observed relationships are decision-support signals, not proof that content changes cause ranking improvements.

In [ ]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Set up Hugging Face authentication without placing the token
# directly inside the SQL statement.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Use February as the feature window and March as the outcome window.
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("DuckDB connected.")
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Warehouse:", REL)

DuckDB connected.
Feature window: February 2026
Outcome window: March 2026
Warehouse: hf://datasets/FlyRank/internship-warehouse


In [ ]:
# Inspect the February feature-window partition
feb_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FEB}
""").df()

print("February 2026 — Feature Window")
display(feb_summary)



# Inspect the March outcome-window partition
mar_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MAR}
""").df()

print("March 2026 — Outcome Window")
display(mar_summary)

February 2026 — Feature Window


,total_rows,unique_content_items,unique_clients,first_date,last_date
0,7355108,321546,54,2026-02-01,2026-02-28


March 2026 — Outcome Window


,total_rows,unique_content_items,unique_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification: Features, Labels, Context, and Excluded Fields

#### 1. Features

Features are variables available by the decision cutoff of 28 February 2026.

**February performance features:**
- `gsc_impressions`
- `gsc_clicks`
- `gsc_sum_position`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`
- `scroll_events`

These are candidate features, subject to data-availability checks. GA4 metrics must only be interpreted where `ga4_data_available` indicates that GA4 data is available. Zero-filled records must not automatically be treated as genuine zero activity.

**Content metadata features:**
- `keyword_char_count`
- `keyword_token_count`
- `url_char_count`
- `content_type`
- `search_volume`
- `competition`
- `competition_level`
- `cpc`
- `main_intent`
- `backlinks`
- `category_count`
- `char_count`
- `word_count`

Content metadata will only be used when it is appropriate for the intended decision and available without future-information leakage. Dates and status fields require additional checks before being used as predictive features.

#### 2. Labels

The label describes the outcome observed after the decision cutoff.

For this analysis, March 2026 will be the outcome window. Candidate outcome measures are:

- **March CTR:** Total March GSC clicks divided by total March GSC impressions.
- **March average search position:** Total March GSC position sum divided by total March GSC impressions.

Content items must have at least 100 March GSC impressions to qualify for the outcome analysis. Outcomes must be calculated from valid GSC records, and position must be handled carefully when data is unavailable.

These are outcome measures, not February predictive features. A separate label definition is required if the final task calls for a binary classification target.

#### 3. Context

Context fields support grouping, interpretation, data-quality checks, or eligibility decisions.

- `report_date` and `month`: define the observation period.
- `client_hash_id`: identifies the client group.
- `content_hash_id`: identifies the content item.
- `keyword_hash_id` and `url_hash_id`: provide hashed reference identifiers.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available`: describe access or data availability.
- `is_active`, `has_gsc_access`, and `has_ga4_access`: describe client status and access.
- `client_created_date`, `client_updated_date`, `gsc_data_start`, and `ga4_data_start`: help assess client history and data coverage.
- `content_created_date`, `content_updated_date`, `keyword_created_date`, `last_optimized_date`, and `optimization_eligible_date`: provide content lifecycle context.
- `is_published` and `is_deleted`: support eligibility and quality checks.
- `provider_used` and `model_used`: provide provenance context.

Identifiers will not be used as ordinary predictive features. Context fields may support filtering or grouping, but their values must be checked for temporal validity.

#### 4. Excluded Fields

The following must be excluded from predictive features unless a separate, documented justification establishes that their use is valid:

- March performance metrics when predicting or prioritizing decisions made by 28 February 2026, because they belong to the outcome window.
- `report_date` and `month` as direct predictive features when they merely encode the observation period.
- Client, content, keyword, and URL identifiers as ordinary predictive features, because they can encourage memorization rather than generalizable learning.
- `provider_used` and `model_used` as predictive features until their timing and potential relationship to downstream outcomes are understood.
- `last_optimized_date` and `content_updated_date` as predictive features unless their values are verified to represent information available by the decision cutoff.
- `is_published`, `is_deleted`, and other status fields when their historical values at the decision cutoff cannot be established.
- Any metric that is unavailable, zero-filled because of missing source access, or derived from future information.

Derived outcomes, including March CTR and March average search position, must never be reused as February features.

The final feature set will be confirmed only after the queries below verify the data grain, coverage, missingness, and temporal validity.

In [ ]:
print("DAILY PERFORMANCE TABLE")
display(
    con.execute(f"""
        DESCRIBE SELECT * FROM {FEB}
    """).df()
)

# Inspect the content dimension
print("CONTENT DIMENSION")
display(
    con.execute(f"""
        DESCRIBE
        SELECT * FROM read_parquet('{DIM_CONTENT}')
    """).df()
)

# Inspect the client dimension
print("CLIENT DIMENSION")
display(
    con.execute(f"""
        DESCRIBE
        SELECT * FROM read_parquet('{DIM_CLIENTS}')
    """).df()
)

DAILY PERFORMANCE TABLE


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


CONTENT DIMENSION


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


CLIENT DIMENSION


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [ ]:
# Check the content dimension's row count and key uniqueness
content_key_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(*) - COUNT(content_hash_id) AS null_content_ids,
        COUNT(*) - COUNT(DISTINCT content_hash_id) AS duplicate_content_key_rows
    FROM read_parquet('{DIM_CONTENT}')
""").df()

print("Content dimension — key and grain check")
display(content_key_check)


# Check whether a content ID is associated with multiple clients
content_client_check = con.execute(f"""
    SELECT
        COUNT(*) AS content_client_pairs,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM (
        SELECT DISTINCT
            content_hash_id,
            client_hash_id
        FROM read_parquet('{DIM_CONTENT}')
        WHERE content_hash_id IS NOT NULL
    )
""").df()

print("Content-to-client mapping check")
display(content_client_check)

Content dimension — key and grain check


,total_rows,unique_content_ids,unique_clients,null_content_ids,duplicate_content_key_rows
0,519606,519606,84,0,0


Content-to-client mapping check


,content_client_pairs,unique_content_ids
0,519606,519606


In [ ]:
# Summarize March GSC availability and outcome eligibility.
# Aggregate by content first, then summarize the smaller result.

march_content = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) AS daily_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
        ) AS gsc_available_days,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS march_impressions,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS march_clicks,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS march_position_sum,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
              AND gsc_impressions IS NULL
        ) AS missing_impression_days,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
              AND gsc_clicks IS NULL
        ) AS missing_click_days,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
              AND gsc_sum_position IS NULL
        ) AS missing_position_sum_days

    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
""").df()

print("Content-level March GSC records:", len(march_content))
display(march_content.head())

Content-level March GSC records: 331437


,client_hash_id,content_hash_id,daily_rows,gsc_available_days,march_impressions,march_clicks,march_position_sum,missing_impression_days,missing_click_days,missing_position_sum_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,31,31,6523.0,7.0,44965.0,0,0,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,31,31,453.0,0.0,1456.0,0,0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,31,31,5630.0,6.0,36794.0,0,0,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,31,31,4944.0,13.0,36762.0,0,0,0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,31,21,42.0,0.0,663.0,0,0,0


In [ ]:
# Summarize the aggregated March outcome records.

march_eligibility = con.execute("""
    SELECT
        COUNT(*) AS content_items,

        COUNT(*) FILTER (
            WHERE gsc_available_days > 0
        ) AS items_with_gsc_data,

        COUNT(*) FILTER (
            WHERE gsc_available_days > 0
              AND missing_impression_days = 0
              AND march_impressions >= 100
        ) AS items_meeting_impression_threshold,

        COUNT(*) FILTER (
            WHERE gsc_available_days = 0
        ) AS items_without_gsc_data,

        COUNT(*) FILTER (
            WHERE missing_impression_days > 0
        ) AS items_with_missing_impressions,

        COUNT(*) FILTER (
            WHERE missing_click_days > 0
        ) AS items_with_missing_clicks,

        COUNT(*) FILTER (
            WHERE missing_position_sum_days > 0
        ) AS items_with_missing_position_sums

    FROM march_content
""").df()

display(march_eligibility)

,content_items,items_with_gsc_data,items_meeting_impression_threshold,items_without_gsc_data,items_with_missing_impressions,items_with_missing_clicks,items_with_missing_position_sums
0,331437,176738,101441,154699,0,0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verified March outcome coverage
- The March 2026 performance partition contains 9,841,378 daily records covering 331,437 unique content items and 55 clients.
- Of the 331,437 content items, 176,738 have at least one day with available GSC data.
- A total of 101,441 content items meet the initial eligibility requirement of at least 100 March GSC impressions.
- Another 154,699 items have no days marked as having available GSC data.
- The eligibility query detected no missing impression, click, or position-sum values on days marked as GSC-available.
- GSC-unavailable observations must not automatically be interpreted as zero performance.
- These counts describe the observed warehouse data; they do not establish that the eligible items are representative of all content.

In [ ]:
# Calculate March outcome measures for eligible content items.
# march_content is the content-level DataFrame created in Step 2.

march_labels = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        march_impressions,
        march_clicks,
        march_position_sum,

        march_clicks / NULLIF(march_impressions, 0)
            AS march_ctr,

        march_position_sum / NULLIF(march_impressions, 0)
            AS march_avg_position

    FROM march_content

    WHERE gsc_available_days > 0
      AND missing_impression_days = 0
      AND missing_click_days = 0
      AND missing_position_sum_days = 0
      AND march_impressions >= 100
""").df()

print("Eligible content items with calculated outcomes:", len(march_labels))
display(march_labels.head())

Eligible content items with calculated outcomes: 101441


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_position_sum,march_ctr,march_avg_position
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,44965.0,0.001073,6.893301
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,1456.0,0.000000,3.214128
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,36794.0,0.001066,6.535346
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,36762.0,0.002629,7.435680
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,1661.0,0.002331,3.871795


### March outcome calculations

The March outcome query produced 101,441 eligible content items after applying the initial requirement of at least 100 GSC impressions and excluding records that failed the GSC availability and missing-value checks.

For each eligible content item:

- March CTR was calculated as total March GSC clicks divided by total March GSC impressions.
- March average position was calculated as total March GSC position sum divided by total March GSC impressions.

The calculated CTR values ranged from 0 to approximately 0.1558, with a mean of approximately 0.002616. Expressed as percentages, the mean item-level CTR was approximately 0.262%.

The mean of the calculated content-level average-position values was approximately 14.35. Neither outcome measure had null values among the eligible records.

These figures describe the observed eligible population. The mean item-level CTR is not equivalent to aggregate CTR across all impressions. The interpretation of the position-sum field must also be confirmed against the warehouse's data definition before finalizing the contract.

In [ ]:
# Summarize the calculated outcome measures.

outcome_summary = con.execute("""
    SELECT
        COUNT(*) AS eligible_items,

        MIN(march_impressions) AS minimum_impressions,
        MAX(march_impressions) AS maximum_impressions,

        AVG(march_ctr) AS mean_march_ctr,
        MIN(march_ctr) AS minimum_march_ctr,
        MAX(march_ctr) AS maximum_march_ctr,

        AVG(march_avg_position) AS mean_march_avg_position,
        MIN(march_avg_position) AS minimum_march_avg_position,
        MAX(march_avg_position) AS maximum_march_avg_position,

        COUNT(*) FILTER (
            WHERE march_ctr IS NULL
        ) AS null_ctr_count,

        COUNT(*) FILTER (
            WHERE march_avg_position IS NULL
        ) AS null_position_count

    FROM march_labels
""").df()

display(outcome_summary)

,eligible_items,minimum_impressions,maximum_impressions,mean_march_ctr,minimum_march_ctr,maximum_march_ctr,mean_march_avg_position,minimum_march_avg_position,maximum_march_avg_position,null_ctr_count,null_position_count
0,101441,100.0,617124.0,0.002616,0.0,0.155844,14.348876,0.015873,106.890995,0,0


In [ ]:
# Aggregate February performance to one row per client and content item.
# GA4 values are only counted when GA4 data is marked available.

feb_features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(*) AS feb_daily_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
        ) AS feb_gsc_available_days,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feb_gsc_impressions,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feb_gsc_clicks,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS feb_gsc_position_sum,

        SUM(
            CASE
                WHEN gsc_data_available = TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feb_impressions_for_position,

        COUNT(*) FILTER (
            WHERE ga4_data_available = TRUE
        ) AS feb_ga4_available_days,

        SUM(
            CASE
                WHEN ga4_data_available = TRUE
                THEN COALESCE(ga4_sessions, 0)
                ELSE 0
            END
        ) AS feb_ga4_sessions,

        SUM(
            CASE
                WHEN ga4_data_available = TRUE
                THEN COALESCE(sessions_organic, 0)
                ELSE 0
            END
        ) AS feb_organic_sessions,

        SUM(
            CASE
                WHEN ga4_data_available = TRUE
                THEN COALESCE(sessions_ai, 0)
                ELSE 0
            END
        ) AS feb_ai_sessions

    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
""").df()

print("February content-level feature records:", len(feb_features))
display(feb_features.head())

February content-level feature records: 321546


,client_hash_id,content_hash_id,feb_daily_rows,feb_gsc_available_days,feb_gsc_impressions,feb_gsc_clicks,feb_gsc_position_sum,feb_impressions_for_position,feb_ga4_available_days,feb_ga4_sessions,feb_organic_sessions,feb_ai_sessions
0,client_3ffa76342f366962,content_fb84747a57b8b665,10,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
1,client_3ffa76342f366962,content_feccf822ac21326e,10,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
2,client_3ffa76342f366962,content_17cf93c10413ebe9,10,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
3,client_3ffa76342f366962,content_a9905735266f8697,10,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0
4,client_3ffa76342f366962,content_31c34765e7bba2f0,10,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0


In [ ]:
feb_feature_summary = con.execute("""
    SELECT
        COUNT(*) AS content_items,

        COUNT(*) FILTER (
            WHERE feb_gsc_available_days > 0
        ) AS items_with_gsc_data,

        COUNT(*) FILTER (
            WHERE feb_gsc_available_days = 0
        ) AS items_without_gsc_data,

        COUNT(*) FILTER (
            WHERE feb_ga4_available_days > 0
        ) AS items_with_ga4_data,

        COUNT(*) FILTER (
            WHERE feb_ga4_available_days = 0
        ) AS items_without_ga4_data,

        MIN(feb_daily_rows) AS minimum_daily_rows,
        MAX(feb_daily_rows) AS maximum_daily_rows

    FROM feb_features
""").df()

display(feb_feature_summary)

,content_items,items_with_gsc_data,items_without_gsc_data,items_with_ga4_data,items_without_ga4_data,minimum_daily_rows,maximum_daily_rows
0,321546,153559,167987,33422,288124,1,28


In [ ]:
# Register the existing aggregated DataFrames with DuckDB
con.register("feb_features_df", feb_features)
con.register("march_labels_df", march_labels)

# Find content items present in both February and March
period_overlap = con.execute("""
    SELECT
        COUNT(*) AS matched_rows,
        COUNT(DISTINCT f.content_hash_id) AS matched_content_items,
        COUNT(DISTINCT f.client_hash_id) AS matched_clients,

        COUNT(*) FILTER (
            WHERE f.feb_gsc_available_days > 0
        ) AS matched_items_with_feb_gsc,

        COUNT(*) FILTER (
            WHERE f.feb_ga4_available_days > 0
        ) AS matched_items_with_feb_ga4,

        COUNT(*) FILTER (
            WHERE f.feb_gsc_available_days = 0
              AND f.feb_ga4_available_days = 0
        ) AS matched_items_without_feb_gsc_or_ga4

    FROM feb_features_df AS f

    INNER JOIN march_labels_df AS m
        ON f.client_hash_id = m.client_hash_id
       AND f.content_hash_id = m.content_hash_id
""").df()

display(period_overlap)

,matched_rows,matched_content_items,matched_clients,matched_items_with_feb_gsc,matched_items_with_feb_ga4,matched_items_without_feb_gsc_or_ga4
0,93671,93671,38,86560,20417,6846


In [ ]:
# Verify the grain and eligibility of the matched dataset
matched_grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        COUNT(DISTINCT client_hash_id) AS unique_clients,

        COUNT(*) - COUNT(DISTINCT content_hash_id)
            AS duplicate_content_ids,

        COUNT(*) FILTER (
            WHERE march_impressions < 100
        ) AS rows_below_impression_threshold

    FROM (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            m.march_impressions

        FROM feb_features_df AS f

        INNER JOIN march_labels_df AS m
            ON f.client_hash_id = m.client_hash_id
           AND f.content_hash_id = m.content_hash_id
    ) AS matched
""").df()

display(matched_grain_check)

,total_rows,unique_content_items,unique_clients,duplicate_content_ids,rows_below_impression_threshold
0,93671,93671,38,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations

**1. Uneven historical coverage**

The February and March partitions contain different numbers of content items and clients. In February, 153,559 content items had at least one day with available GSC data, while 33,422 had at least one day with available GA4 data. Some records contain fewer daily observations than others. Consequently, the available data does not provide equally complete historical coverage for every content item.

**2. Uneven source availability**

GA4 coverage is substantially lower than GSC coverage in the February window. Missing or unavailable GA4 observations must not be interpreted as zero activity. Analyses using GA4 metrics may represent a smaller and potentially different population from analyses using GSC metrics.

**3. Outcome eligibility and selection**

Only 101,441 content items met the initial requirement of at least 100 March GSC impressions and the specified availability checks. These eligible items may differ from content items without sufficient GSC data. Results from the eligible population should not automatically be generalized to all content.

**4. Temporal limitations**

February is the feature window, and March is the outcome window. March performance must not be used as a predictive feature for a decision made by the end of February. Although the date windows are separate, the same content item can appear in both periods. This overlap is intentional because February observations are used to study subsequent March outcomes.

**5. Observational data cannot establish causation**

The warehouse records observed performance and content characteristics. It cannot, by itself, establish that updating a page caused a change in clicks, CTR, or search position. Other factors may influence observed outcomes, including competition, demand, seasonality, and changes in search systems.

**6. Historical metadata limitations**

Current content and client metadata may not represent the exact state of a page or client on 28 February 2026. Fields describing optimization, publication, deletion, or update status should not be used as historical predictive features unless their values at the decision cutoff can be verified.

**7. Interpretation of search metrics**

CTR and average position describe different aspects of search performance. Item-level averages can differ from aggregate metrics across all impressions. Position calculations also depend on the meaning and validity of the underlying position-sum field, which should be confirmed against the warehouse's data definition.

**Intended use:** The analysis can support editorial-review prioritization by identifying observed patterns and potential opportunities. It should be treated as decision support, not as proof of causation or a guarantee of future performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.